In [23]:
import os
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages

load_dotenv()

True

In [24]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("groq:llama-3.3-70b-versatile")

In [25]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

def chatbot(state: State) -> State:
   return {"messages": [llm.invoke(state["messages"])]}

In [26]:
from langgraph.graph import StateGraph, START, END


builder = StateGraph(State)

builder.add_node("chatbot_node", chatbot)

builder.add_edge(START, "chatbot_node")
builder.add_edge("chatbot_node", END)

graph = builder.compile()

In [27]:
message={"role": "user", "content": "Who walked on the moon for the first time print only the name"}
response = graph.invoke({"messages": [message]})

print(response["messages"])

[HumanMessage(content='Who walked on the moon for the first time print only the name', additional_kwargs={}, response_metadata={}, id='c0c1a190-759f-4318-a241-66d7fa9dcfee'), AIMessage(content='Neil Armstrong', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 3, 'prompt_tokens': 48, 'total_tokens': 51, 'completion_time': 0.007431618, 'completion_tokens_details': None, 'prompt_time': 0.001718634, 'prompt_tokens_details': None, 'queue_time': 0.049797084, 'total_time': 0.009150252}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019c75ff-88aa-73a1-835b-0c792005444f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 48, 'output_tokens': 3, 'total_tokens': 51})]


In [32]:
#loop
state = None
while True:
    in_message = input("You: ")
    if in_message.lower() in ["exit", "quit"]:
        break

    if state is None:
        state: State = {
            "messages": [{"role": "user", "content": in_message}]
        }
    else:
        state["messages"].append({"role": "user", "content": in_message})

    state = graph.invoke(state)
    print("Bot:", state["messages"][-1].content)

Bot: Neil Armstrong was the first person to walk on the Moon. He stepped out of the lunar module Eagle and onto the Moon's surface on July 20, 1969, during the Apollo 11 mission. Armstrong famously declared, "That's one small step for man, one giant leap for mankind," as he became the first human to set foot on the Moon.
Bot: 1969
Bot: Neil Armstrong walked on the Moon with Edwin "Buzz" Aldrin. They were the first two people to set foot on the Moon's surface during the Apollo 11 mission in 1969. Michael Collins remained in orbit around the Moon in the command module Columbia.


In [33]:
state["messages"]

[HumanMessage(content='who walked in moon first?', additional_kwargs={}, response_metadata={}, id='5e22fb5a-14a6-4679-b9fb-6bc4dd08c048'),
 AIMessage(content='Neil Armstrong was the first person to walk on the Moon. He stepped out of the lunar module Eagle and onto the Moon\'s surface on July 20, 1969, during the Apollo 11 mission. Armstrong famously declared, "That\'s one small step for man, one giant leap for mankind," as he became the first human to set foot on the Moon.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 75, 'prompt_tokens': 41, 'total_tokens': 116, 'completion_time': 0.145279113, 'completion_tokens_details': None, 'prompt_time': 0.00134958, 'prompt_tokens_details': None, 'queue_time': 0.049043758, 'total_time': 0.146628693}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_68f543a7cc', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019c7642-5a35-7f80-bc50-